# Step 1 Retuned: TX-Only Coincidence + Optuna


This notebook retunes the TX-delayed coincidence architecture and adds calibration/per-bin analyses.


In [ ]:

# =============================================
# Step 1 Retuned
# TX-delay-only coincidence SNN + Optuna tuning
# =============================================
# This notebook is intentionally heavily commented so every part of the
# pipeline is clear and easy to modify.

from pathlib import Path
import json
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import optuna
import snntorch as snn


# -----------------------------
# Reproducibility and artifacts
# -----------------------------
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Artifact directories for this step.
ART = Path("artifacts_steps/step1_retuned_optuna")
PLOT_DIR = ART / "plots"
CKPT_DIR = ART / "checkpoints"
PARAM_DIR = ART / "params"
OPTUNA_DIR = ART / "optuna"

for d in [ART, PLOT_DIR, CKPT_DIR, PARAM_DIR, OPTUNA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CKPT_DIR / "model_step1_retuned.pt"
BEST_PARAMS_PATH = PARAM_DIR / "best_params_step1_retuned.json"
METRICS_PATH = ART / "metrics_step1_retuned.json"
OPTUNA_DB_PATH = OPTUNA_DIR / "step1_retuned.sqlite3"

print("Artifacts:", ART.resolve())


# -------------------------
# Device selection (CPU/GPU)
# -------------------------
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("DEVICE:", DEVICE)


# ------------------------------
# Experiment control and toggles
# ------------------------------
# If RUN_OPTUNA=True, the notebook performs a hyperparameter search first.
RUN_OPTUNA = True
N_TRIALS = 16
TRIAL_EPOCHS = 12

# Final training pass with selected params.
FINAL_EPOCHS = 30

# Load behavior:
# - if a checkpoint exists and FORCE_RETRAIN=False, we skip full retraining.
FORCE_RETRAIN = False
LOAD_CHECKPOINT_IF_AVAILABLE = True


# --------------------------------------
# Signal simulation and dataset settings
# --------------------------------------
DT = 1e-4
T_STEPS = 768
TIME = np.arange(T_STEPS) * DT
SPEED_OF_SOUND = 343.0

DISTANCE_CLASSES_M = np.arange(0.5, 10.5, 0.5)
NUM_CLASSES = len(DISTANCE_CLASSES_M)

# Sample duplication count per class. Higher => more data, slower training.
SAMPLES_PER_CLASS = 30

# Basic pulse/echo model parameters.
PULSE_CENTER = 35
PULSE_STD = 6.0
ECHO_ATT = 0.55

# Delay bank taps for TX-only delay path.
DELAY_STEPS = torch.arange(0, 673, 12).long()
assert int(DELAY_STEPS.max().item()) < T_STEPS

# Fallback/default params when no tuning JSON exists yet.
DEFAULT_PARAMS = {
    "hidden_size": 160,
    "beta": 0.95,
    "lr": 8e-4,
    "batch_size": 64,
}


# ------------------
# Utility functions
# ------------------
def save_fig(name: str):
    path = PLOT_DIR / name
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved plot:", path)


def save_json(path: Path, payload: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def load_json(path: Path):
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# --------------------------------
# Waveform simulation (TX and RX)
# --------------------------------
def gaussian_pulse():
    """Create a simple Gaussian pulse used as the transmitted template."""
    idx = np.arange(T_STEPS)
    return np.exp(-0.5 * ((idx - PULSE_CENTER) / PULSE_STD) ** 2).astype(np.float32)


def make_sample(distance_m: float):
    """
    Build one ideal TX/RX pair:
    - TX: Gaussian pulse
    - RX: delayed and attenuated copy of TX using round-trip time delay
    """
    tx = gaussian_pulse()
    delay_steps = int(round((2.0 * distance_m / SPEED_OF_SOUND) / DT))

    rx = np.zeros_like(tx)
    if delay_steps < T_STEPS:
        rx[delay_steps:] = ECHO_ATT * tx[: T_STEPS - delay_steps]

    return tx.astype(np.float32), rx.astype(np.float32), delay_steps


# ---------------------
# Dataset construction
# ---------------------
def build_dataset(seed=SEED):
    rng = np.random.default_rng(seed)

    txs = []
    rxs = []
    ys = []

    for ci, d in enumerate(DISTANCE_CLASSES_M):
        tx, rx, _ = make_sample(float(d))
        for _ in range(SAMPLES_PER_CLASS):
            txs.append(tx)
            rxs.append(rx)
            ys.append(ci)

    txs = np.stack(txs)
    rxs = np.stack(rxs)
    ys = np.array(ys, dtype=np.int64)

    perm = rng.permutation(len(ys))
    txs, rxs, ys = txs[perm], rxs[perm], ys[perm]

    split = int(0.8 * len(ys))
    return txs[:split], rxs[:split], ys[:split], txs[split:], rxs[split:], ys[split:]


def make_loaders(Xtx_tr, Xrx_tr, y_tr, Xtx_te, Xrx_te, y_te, batch_size):
    train_ds = TensorDataset(
        torch.from_numpy(Xtx_tr),
        torch.from_numpy(Xrx_tr),
        torch.from_numpy(y_tr),
    )
    test_ds = TensorDataset(
        torch.from_numpy(Xtx_te),
        torch.from_numpy(Xrx_te),
        torch.from_numpy(y_te),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_ds, test_ds, train_loader, test_loader


# -----------------------------
# Model: TX-only delay bank SNN
# -----------------------------
class DelayBank(nn.Module):
    """
    Fixed delay bank.
    Input:  [T, B, 1]
    Output: [T, B, D]
    where D is number of delay taps.
    """

    def __init__(self, delays):
        super().__init__()
        self.register_buffer("delays", delays.clone().long())

    def forward(self, x):
        T, B, _ = x.shape
        D = self.delays.numel()
        out = torch.zeros(T, B, D, device=x.device, dtype=x.dtype)
        x_flat = x[:, :, 0]

        # Delay each channel independently by its fixed tap.
        for i, d in enumerate(self.delays.tolist()):
            if d == 0:
                out[:, :, i] = x_flat
            elif d < T:
                out[d:, :, i] = x_flat[: T - d, :]

        return out


class CoincidenceSNN(nn.Module):
    """
    Coincidence model:
    1) Delay TX only
    2) Coincidence = delayed_tx * rx
    3) Hidden LIF layer
    4) Output LIF layer for class spikes
    """

    def __init__(self, delays, hidden_size=160, beta=0.95):
        super().__init__()
        self.delay = DelayBank(delays)

        self.fc1 = nn.Linear(len(delays), hidden_size)
        self.lif1 = snn.Leaky(beta=beta)

        self.fc2 = nn.Linear(hidden_size, NUM_CLASSES)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, tx, rx):
        # Delay only transmitted stream.
        dtx = self.delay(tx)

        # Coincidence interaction with received stream.
        coinc = dtx * rx

        self.lif1.reset_mem()
        self.lif2.reset_mem()

        mem1 = None
        mem2 = None
        out_rec = []

        for t in range(coinc.shape[0]):
            s1, mem1 = self.lif1(self.fc1(coinc[t]), mem1)
            s2, mem2 = self.lif2(self.fc2(s1), mem2)
            out_rec.append(s2)

        return torch.stack(out_rec, dim=0)


# -------------------------
# Evaluation and training
# -------------------------
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    y_true = []
    y_pred = []
    y_conf = []

    with torch.no_grad():
        for txb, rxb, yb in loader:
            txb = txb.to(DEVICE)
            rxb = rxb.to(DEVICE)
            yb = yb.to(DEVICE)

            tx_t = txb.unsqueeze(-1).permute(1, 0, 2)
            rx_t = rxb.unsqueeze(-1).permute(1, 0, 2)

            spk = model(tx_t, rx_t)
            cnt = spk.sum(dim=0)

            loss = criterion(cnt, yb)
            total_loss += loss.item() * yb.size(0)

            probs = torch.softmax(cnt, dim=1)
            conf, pred = probs.max(dim=1)

            correct += (pred == yb).sum().item()
            total += yb.size(0)

            y_true.append(yb.cpu().numpy())
            y_pred.append(pred.cpu().numpy())
            y_conf.append(conf.cpu().numpy())

    return {
        "loss": total_loss / total,
        "acc": correct / total,
        "y_true": np.concatenate(y_true),
        "y_pred": np.concatenate(y_pred),
        "y_conf": np.concatenate(y_conf),
    }


def train_model(model, train_loader, test_loader, lr, epochs, verbose=True):
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [],
        "test_loss": [],
        "train_acc": [],
        "test_acc": [],
    }

    best_state = None
    best_test_loss = float("inf")

    for ep in range(1, epochs + 1):
        model.train()
        run_loss = 0.0
        run_cor = 0
        run_tot = 0

        for txb, rxb, yb in train_loader:
            txb = txb.to(DEVICE)
            rxb = rxb.to(DEVICE)
            yb = yb.to(DEVICE)

            tx_t = txb.unsqueeze(-1).permute(1, 0, 2)
            rx_t = rxb.unsqueeze(-1).permute(1, 0, 2)

            opt.zero_grad()
            spk = model(tx_t, rx_t)
            cnt = spk.sum(dim=0)
            loss = criterion(cnt, yb)
            loss.backward()
            opt.step()

            run_loss += loss.item() * yb.size(0)
            pred = cnt.argmax(dim=1)
            run_cor += (pred == yb).sum().item()
            run_tot += yb.size(0)

        tr_loss = run_loss / run_tot
        tr_acc = run_cor / run_tot

        te_eval = evaluate(model, test_loader, criterion)
        te_loss = te_eval["loss"]
        te_acc = te_eval["acc"]

        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["train_acc"].append(tr_acc)
        history["test_acc"].append(te_acc)

        if te_loss < best_test_loss:
            best_test_loss = te_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if verbose and (ep == 1 or ep % 5 == 0 or ep == epochs):
            print(
                f"Epoch {ep:02d}/{epochs} | "
                f"train_loss={tr_loss:.4f}, train_acc={tr_acc*100:.1f}% | "
                f"test_loss={te_loss:.4f}, test_acc={te_acc*100:.1f}%"
            )

    return history, best_state, best_test_loss


# -------------------------------
# Calibration and error analyses
# -------------------------------
def plot_calibration(y_true, y_pred, y_conf, filename="calibration_step1_retuned.png", n_bins=10):
    """
    Reliability diagram:
    - bin by confidence
    - compare average confidence vs empirical accuracy in each bin
    """
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.digitize(y_conf, bins[1:-1], right=False)

    accs = []
    confs = []
    counts = []

    for b in range(n_bins):
        m = bin_idx == b
        if np.any(m):
            accs.append(np.mean(y_pred[m] == y_true[m]))
            confs.append(np.mean(y_conf[m]))
            counts.append(np.sum(m))
        else:
            accs.append(np.nan)
            confs.append(np.nan)
            counts.append(0)

    # ECE (expected calibration error)
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        if counts[b] > 0:
            ece += (counts[b] / n) * abs(accs[b] - confs[b])

    centers = 0.5 * (bins[:-1] + bins[1:])

    plt.figure(figsize=(6.5, 5.0))
    plt.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Perfect calibration")
    plt.plot(centers, accs, marker="o", linewidth=2, label="Observed accuracy")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(f"Calibration curve (ECE={ece:.4f})")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    save_fig(filename)
    plt.show()

    return float(ece)


def plot_per_bin_error(y_true, y_pred, filename="per_bin_error_step1_retuned.png"):
    """
    Per-distance-bin error histogram.
    Shows where the classifier struggles across true distance bins.
    """
    error_rate = []
    for c in range(NUM_CLASSES):
        m = y_true == c
        if np.any(m):
            error_rate.append(float(np.mean(y_pred[m] != y_true[m])))
        else:
            error_rate.append(np.nan)

    plt.figure(figsize=(10, 4.5))
    plt.bar(np.arange(NUM_CLASSES), error_rate, color="#4C72B0")
    plt.xticks(np.arange(NUM_CLASSES), [f"{d:.1f}" for d in DISTANCE_CLASSES_M], rotation=90)
    plt.xlabel("True distance bin (m)")
    plt.ylabel("Error rate")
    plt.title("Per-bin classification error")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    save_fig(filename)
    plt.show()


# ----------------------------------
# Data prep and baseline signal plot
# ----------------------------------
Xtx_tr, Xrx_tr, y_tr, Xtx_te, Xrx_te, y_te = build_dataset()
print("Train:", len(y_tr), "Test:", len(y_te), "Chance:", 100.0 / NUM_CLASSES)

# Quick signal visualization.
example_tx, example_rx, example_delay = make_sample(float(DISTANCE_CLASSES_M[10]))
plt.figure(figsize=(10, 4))
plt.plot(TIME * 1e3, example_tx, label="TX pulse")
plt.plot(TIME * 1e3, example_rx, label="RX echo")
plt.title(f"Step1 retuned sample (delay={example_delay} steps)")
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
save_fig("signal_example_step1_retuned.png")
plt.show()


# ----------------
# Optuna objective
# ----------------
def objective(trial):
    random.seed(SEED + trial.number)
    np.random.seed(SEED + trial.number)
    torch.manual_seed(SEED + trial.number)

    params = {
        "hidden_size": trial.suggest_categorical("hidden_size", [96, 128, 160, 192, 256]),
        "beta": trial.suggest_float("beta", 0.88, 0.99),
        "lr": trial.suggest_float("lr", 2e-4, 3e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
    }

    _, _, tr_loader, te_loader = make_loaders(
        Xtx_tr, y_tr*0 + y_tr, y_tr,  # placeholder, replaced below
        Xtx_te, y_te*0 + y_te, y_te,
        params["batch_size"],
    )

    # The loader utility expects tx/rx/label arrays. Build directly with proper arrays.
    train_ds = TensorDataset(
        torch.from_numpy(Xtx_tr), torch.from_numpy(Xrx_tr), torch.from_numpy(y_tr)
    )
    test_ds = TensorDataset(
        torch.from_numpy(Xtx_te), torch.from_numpy(Xrx_te), torch.from_numpy(y_te)
    )
    tr_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True)
    te_loader = DataLoader(test_ds, batch_size=params["batch_size"], shuffle=False)

    model_t = CoincidenceSNN(
        DELAY_STEPS,
        hidden_size=params["hidden_size"],
        beta=params["beta"],
    ).to(DEVICE)

    hist_t, _, best_test_loss = train_model(
        model_t,
        tr_loader,
        te_loader,
        lr=params["lr"],
        epochs=TRIAL_EPOCHS,
        verbose=False,
    )

    max_train = float(max(hist_t["train_acc"]))
    max_test = float(max(hist_t["test_acc"]))
    trial.set_user_attr("max_train_acc", max_train)
    trial.set_user_attr("max_test_acc", max_test)

    # Stop Optuna as soon as we hit perfect train+test in any trial.
    if max_train >= 0.999 and max_test >= 0.999:
        trial.set_user_attr("perfect_accuracy", True)
        trial.study.set_user_attr("perfect_params", params)
        trial.study.set_user_attr("perfect_trial_number", trial.number)
        trial.study.stop()
        return -1e-6

    return best_test_loss


# ---------------------
# Select hyperparameters
# ---------------------
if RUN_OPTUNA:
    storage = f"sqlite:///{OPTUNA_DB_PATH.resolve()}"
    study = optuna.create_study(
        study_name="step1_retuned_optuna",
        storage=storage,
        direction="minimize",
        load_if_exists=True,
    )
    study.optimize(objective, n_trials=N_TRIALS)

    perfect_params = study.user_attrs.get("perfect_params", None)
    if perfect_params is not None:
        best_params = perfect_params
        print("Optuna stopped early on perfect accuracy trial.")
        print("Perfect trial number:", study.user_attrs.get("perfect_trial_number"))
    else:
        best_params = study.best_params

    print("Selected params:", best_params)

    save_json(
        BEST_PARAMS_PATH,
        {
            "params": best_params,
            "metadata": {
                "storage": storage,
                "study_name": "step1_retuned_optuna",
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "used_perfect_early_stop": perfect_params is not None,
                "perfect_trial_number": study.user_attrs.get("perfect_trial_number", None),
            },
        },
    )
else:
    loaded = load_json(BEST_PARAMS_PATH)
    best_params = loaded["params"] if loaded is not None else DEFAULT_PARAMS.copy()
    print("Using params:", best_params)


# -----------------------------
# Final training / load section
# -----------------------------
train_ds = TensorDataset(
    torch.from_numpy(Xtx_tr), torch.from_numpy(Xrx_tr), torch.from_numpy(y_tr)
)
test_ds = TensorDataset(
    torch.from_numpy(Xtx_te), torch.from_numpy(Xrx_te), torch.from_numpy(y_te)
)

train_loader = DataLoader(train_ds, batch_size=best_params["batch_size"], shuffle=True)
test_loader = DataLoader(test_ds, batch_size=best_params["batch_size"], shuffle=False)

model = CoincidenceSNN(
    DELAY_STEPS,
    hidden_size=best_params["hidden_size"],
    beta=best_params["beta"],
).to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params:", trainable_params)

if LOAD_CHECKPOINT_IF_AVAILABLE and CHECKPOINT_PATH.exists() and not FORCE_RETRAIN:
    payload = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(payload["model_state_dict"])
    history = payload["history"]
    print("Loaded checkpoint:", CHECKPOINT_PATH)
else:
    history, best_state, best_test_loss = train_model(
        model,
        train_loader,
        test_loader,
        lr=best_params["lr"],
        epochs=FINAL_EPOCHS,
        verbose=True,
    )
    model.load_state_dict(best_state)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "history": history,
            "params": best_params,
            "trainable_params": trainable_params,
        },
        CHECKPOINT_PATH,
    )
    print("Saved checkpoint:", CHECKPOINT_PATH)


# -----------------------
# Curves and confusion
# -----------------------
epochs = np.arange(1, len(history["train_loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

ax[0].plot(epochs, history["train_loss"], label="Train")
ax[0].plot(epochs, history["test_loss"], label="Test")
ax[0].set_title("Loss")
ax[0].grid(alpha=0.3)
ax[0].legend()

ax[1].plot(epochs, np.array(history["train_acc"]) * 100, label="Train")
ax[1].plot(epochs, np.array(history["test_acc"]) * 100, label="Test")
ax[1].set_title("Accuracy (%)")
ax[1].grid(alpha=0.3)
ax[1].legend()

plt.tight_layout()
save_fig("curves_step1_retuned.png")
plt.show()

ce = nn.CrossEntropyLoss()
te_eval = evaluate(model, test_loader, ce)
y_true = te_eval["y_true"]
y_pred = te_eval["y_pred"]
y_conf = te_eval["y_conf"]

# confusion matrix
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
im0 = axes[0].imshow(cm, cmap="Blues", aspect="auto")
axes[0].set_title("Confusion (counts)")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
im1 = axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1, aspect="auto")
axes[1].set_title("Confusion (row-normalized)")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
labels = [f"{d:.1f}" for d in DISTANCE_CLASSES_M]
for a in axes:
    a.set_xticks(np.arange(NUM_CLASSES))
    a.set_yticks(np.arange(NUM_CLASSES))
    a.set_xticklabels(labels, rotation=90)
    a.set_yticklabels(labels)
    a.set_xlabel("Pred")
    a.set_ylabel("True")
plt.tight_layout()
save_fig("confusion_step1_retuned.png")
plt.show()

# extra analyses
ece = plot_calibration(y_true, y_pred, y_conf)
plot_per_bin_error(y_true, y_pred)

# Save consolidated metrics for auto-summary.
save_json(
    METRICS_PATH,
    {
        "final_train_acc": float(history["train_acc"][-1]),
        "final_test_acc": float(te_eval["acc"]),
        "best_test_loss": float(min(history["test_loss"])),
        "ece": float(ece),
        "trainable_params": int(trainable_params),
        "params": best_params,
        "artifacts": {
            "curves": str(PLOT_DIR / "curves_step1_retuned.png"),
            "confusion": str(PLOT_DIR / "confusion_step1_retuned.png"),
            "calibration": str(PLOT_DIR / "calibration_step1_retuned.png"),
            "per_bin_error": str(PLOT_DIR / "per_bin_error_step1_retuned.png"),
        },
    },
)
print("Saved metrics:", METRICS_PATH)
